# Day 8: RAG Systems — Retrieval-Augmented Generation

Key Vocabulary

chunk - A small piece of a document

Embedding  - A vector representing text meaning

Vector Database - A database that stores and searches vector by similarity

Retrieval - Finding the most relevant chunks for a given query


In [2]:
!pip install sentence-transformers chromadb groq pandas -q

In [3]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq

import os
print('All libraries imported successfully.')
print("Read to build a RAG system.")



All libraries imported successfully.
Read to build a RAG system.


In [ ]:
GROQ_API_KEY ="YOUR_GROQ_API_KEY"
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq API client initialized:")
print("Note: If you see an authentication error later, double-check your API key")

Groq API client initialized:
Note: If you see an authentication error later, double-check your API key


Loading knowledge base

In [5]:
df= pd.read_csv('college_notes.csv')
print("Shape of dataset",df.shape)
print("\nColumn names:",df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

print("Subjects in this dataset:")
print(df['subject'].value_counts())

print("\n Sample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))

print("\n Length of content (number of characters) for each note:")
print()
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Shape of dataset (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']

First 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
Subjects in this dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

 Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering     

# Chunking

In [6]:
documents = df['content'].tolist()

ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]

metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared : {len(documents)}")
print(f"First document ID     : {ids[0]}")
print(f"First metadata        : {metadatas[0]}")
print(f"First 100 chars of doc:  {documents[0][:100]}...")

Total chunks prepared : 15
First document ID     : note_N001
First metadata        : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc:  ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


# Embeddings

In [7]:
print("Loading embedding model...")
print("(This may take 30-60 seconds on first run --model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")



Loading embedding model...
(This may take 30-60 seconds on first run --model is being downloaded)
(Subsequent runs will be faster as the model is cached)


In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("\n Embedding model loaded successfully")
test_embedding = embedding_model.encode("This is a test sentence.")

In [9]:
print("Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of test emnbedding: {test_embedding[:5]}")

Test embedding shape: {test_embedding.shape}
First 5 values of test emnbedding: [0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [10]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")
print("ChromaDB client created.")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far:{collection.count()}")

ChromaDB client created.
Collection name: college_notes_rag
Documents in collection so far:0


In [11]:
print("Generating embeddings for all 15 notes...")
print("This may take 15-30 seconds")

Generating embeddings for all 15 notes...
This may take 15-30 seconds


In [ ]:
embeddings = embedding_model.encode(documents, show_progress_bar = True)
print(f'\n Embedding matrix shape: {embeddings.shape}')
embeddings_list = embeddings.tolist()
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list,
)

In [13]:
print(f"\nDocuments successfully added to ChromaDB.")
print(f"Total documents in collection: {collection.count()}")


Documents successfully added to ChromaDB.
Total documents in collection: 15


In [14]:
def retrieve_relevant_chunks(question, top_k=3):
    question_embedding = embedding_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )
    return results
print("Retrieval function defined successfully.")
print("Function:retrieve_relavant_chunks(question,top_k=3)")

Retrieval function defined successfully.
Function:retrieve_relavant_chunks(question,top_k=3)


In [15]:
test_question = "What is ETL and how does it work in data engineering"
print(f"The Question: {test_question}")
print("=" * 60)
results = retrieve_relevant_chunks(test_question, top_k=3)
print("\nTop 3 Retrieved Chunks:")
print("=" * 60)
for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"\nResult {i+1}:")
    print(f"  Subject : {meta['subject']}")
    print(f"  Topic   : {meta['topic']}")
    print(f"  Distance: {dist:.4f}")
    print(f"  Context : {doc[:120]}...")

The Question: What is ETL and how does it work in data engineering

Top 3 Retrieved Chunks:

Result 1:
  Subject : Data Engineering
  Topic   : ETL Pipelines
  Distance: 0.2041
  Context : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result 2:
  Subject : Data Engineering
  Topic   : APIs and Data Collection
  Distance: 1.1100
  Context : An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result 3:
  Subject : Python Programming
  Topic   : Data Visualization
  Distance: 1.3892
  Context : Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


# Context Injection
SYSTEM:

You are a helpful academic assistant. Answer questions based only on the provided content..
If the answer is not in the context, say "I don't have enough information to answer this."
Do not use you

context:

[Retrieved document 1]

[Retrieved document 2]

[Retrieved document 3]


In [16]:
def build_context_from_results(results):
  context_parts =[]                        # An empty list to collect formatted chunks
  for i, (doc,meta) in enumerate(zip(results['documents'][0],
                                     results['metadatas'][0])):
    chunk_text = f"[Source {i+1}: {meta['subject']}-{meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)
  context_str = "\n\n--\n\n".join(context_parts)
  return context_str
context = build_context_from_results(results)
print("Built context string from retrieved chunks: ")
print("="*60)
print(context[:500]+"...")
print(f"\n Total context length: {len(context)} characters")

Built context string from retrieved chunks: 
[Source 1: Data Engineering-ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

--

[Source 2: Data Engineering-APIs and Data Collection]
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weath...

 Total context length: 848 characters


In [17]:
#build the rag  generation function


def generate_rag_answer(question,context):
  system_prompt = """You are a helpful academic assistant for engineering students.

  You will be given context retrieved froma college knowledge base, and a student's question.

  Rules:
  1.Answer only using the information provided in the context below.
  2.If the answer is not found in the context, say exactly:
     "I don't have enough information in my knowledge base to answer this question."
  3.Do not use your general training knowledge.
  4.Keep answers clear, accurate, and beginner-friendly.
  5.Mention which source the information came from when possible. """

  user_prompt = f"""Context from knowledge Base:

{context}

---

Student's Question: {question}

Please answer the question based only on the context provided above."""

  response = groq_client.chat.completions.create(
      model = "llama-3.1-8b-instant",
      messages = [
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_prompt}
      ],
      temperature=0.1,
      max_tokens =500
  )
  answer = response.choices[0].message.content
  return answer
print("RAG generation function defined.")

RAG generation function defined.


In [18]:
def ask_college_assistant(question,top_k=3,verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("=" * 60)
    print("Step 1: Retrieving most relevant chunks...")
  results = retrieve_relevant_chunks(question,top_k=top_k)

  if verbose:
        print(f"Retrieved {top_k} chunks from the knowledge base:")
        for i, meta in enumerate(results['metadatas'][0]):
            print(f"  {i+1}. {meta['subject']} - {meta['topic']}")
        print("\nStep 2: Building context string...")
  context = build_context_from_results(results)
  if verbose:
    print(f"Context built ({len(context)}) characters")
    print("\nStep 3:Sending to LLM for answer generation...")
  answer = generate_rag_answer(question,context)
  if verbose:
    print("\n" + "=" * 60)
    print("ANSWER:")
    print("=" * 60)
    print(answer)
    print("=" * 60)
  return answer
print("Complete RAG pipeline function ready.")
print("Function:ask_college_assistant(question,top_k=3)")

Complete RAG pipeline function ready.
Function:ask_college_assistant(question,top_k=3)


In [19]:
question_1 = "What is ETL and what are its three main stages?"
answer_1 = ask_college_assistant(question_1,top_k=3,verbose=True)

Question: What is ETL and what are its three main stages?
Step 1: Retrieving most relevant chunks...
Retrieved 3 chunks from the knowledge base:
  1. Data Engineering - ETL Pipelines
  2. Generative AI - Retrieval Augmented Generation
  3. Generative AI - Prompt Engineering

Step 2: Building context string...
Context built (922) characters

Step 3:Sending to LLM for answer generation...

ANSWER:
According to the context from [Source 1: Data Engineering-ETL Pipelines], ETL stands for Extract Transform Load. 

The three main stages of ETL are:

1. Extract: This stage involves collecting raw data from different sources.
2. Transform: This stage involves transforming the raw data into a clean and structured format.
3. Load: This stage involves loading the transformed data into a database or data warehouse for analysis.

Source: [Source 1: Data Engineering-ETL Pipelines]


In [20]:
question_2 = "How do embeddings help in building search systems"
answer_2 = ask_college_assistant(question_2,top_k=3, verbose=True)

Question: How do embeddings help in building search systems
Step 1: Retrieving most relevant chunks...
Retrieved 3 chunks from the knowledge base:
  1. Generative AI - Retrieval Augmented Generation
  2. Generative AI - Large Language Models
  3. Machine Learning - Feature Engineering

Step 2: Building context string...
Context built (902) characters

Step 3:Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.


In [21]:
question_3 = "What is the  population of Tokyo"
print("Testing with an out-of-scope question (not in college notes):")
answer_3 = ask_college_assistant(question_3, top_k=3, verbose =True)

Testing with an out-of-scope question (not in college notes):
Question: What is the  population of Tokyo
Step 1: Retrieving most relevant chunks...
Retrieved 3 chunks from the knowledge base:
  1. Generative AI - Large Language Models
  2. Data Engineering - SQL Databases
  3. Data Engineering - Data Cleaning

Step 2: Building context string...
Context built (791) characters

Step 3:Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.


In [22]:
def retrive_key_subject(question,subject_filter,top_k=3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k,
      where={"subject": subject_filter}
  )
  return results
print("Retrieving only from GenAI subject:")
print("=" * 50)
filtered_results = retrive_key_subject(
    question="How do LLMs generate text?",
    subject_filter="GenAI",
    top_k=3
)
for i, (doc, dist) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
)):
    print(f"Result{i+1}:[{meta['subject']}]{meta['topic']}")
    print(f"{doc[:100]}...")

Retrieving only from GenAI subject:


## Q1. What is hallucination in LLMs?

**Answer:**  
Hallucination is when an LLM generates incorrect or made-up information that sounds convincing.

---

## Q2. What does RAG stand for? What problem does it solve?

**Answer:**  
RAG stands for **Retrieval-Augmented Generation**.

It reduces hallucinations by retrieving relevant information from external documents before generating an answer.

---

## Q3. What is the role of a vector database in the RAG pipeline?

**Answer:**  
A vector database:
- Stores document embeddings
- Performs semantic search
- Retrieves relevant chunks for a query

---

## Q4. Difference between Indexing Phase and Querying Phase

### Indexing Phase
- Load documents
- Create embeddings
- Store in vector database

### Querying Phase
- Convert query to embedding
- Retrieve relevant chunks
- Generate answer using LLM

---

## Q5. Why use the same embedding model for documents and queries?

**Answer:**  
Using the same embedding model ensures documents and queries are represented in the same vector space, resulting in accurate similarity search.

---

## Q6. Why is a low temperature (e.g., 0.1) preferred for RAG?

**Answer:**  
A low temperature:
- Produces consistent answers
- Reduces randomness
- Minimizes hallucinations
- Keeps responses grounded in retrieved documents

## Q7. Display distance scores of retrieved chunks
**Answer:**

In [23]:
for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}")
    print(doc)

Distance: 0.2041
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
Distance: 1.1100
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
Distance: 1.3892
Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts line plots histograms and pie charts that help humans understand patterns in data.


## Q8. Change the system prompt to always respond in bullet points

**Answer:**

In [24]:
system_prompt = """
You are a helpful assistant.
Always answer using bullet points.
Use only the provided context.
"""

## Q9. Return only topic names of retrieved chunks

**Answer:**

In [25]:

def get_topics(results):
    return [meta["topic"] for meta in results["metadatas"][0]]

# MINI PROJECT -College Knowledge Assistant

###Project Description

Build a complete College Knowledge Assistant that:
1. Loads the college_notes.csv knowledge base
2.Indexes all notes in ChromaDB with embeddings
3.Accepts a student Question
4. Retrieves the top 3 relevant notes
5. Injects them as content into a Groq LLM prompt
6. Returns a clear, grounded answer with source citiations
7. Handles questions outside the knowledge base gracefully

1. college_notes.csv



          

2. Load Dataset



          

3. Create Embeddings



4. Store in ChromaDB



5. Student Question



6. Convert Question -> Embedding



7. Similarity Search



8. Top 3 Notes Retrieved



9. Groq LLM



10. Answer + Sources


In [26]:
import pandas as pd

df = pd.read_csv("college_notes.csv")

print(df.shape)
print(df.head())

(15, 4)
  note_id           subject                     topic  \
0    N001  Data Engineering             ETL Pipelines   
1    N002  Data Engineering             SQL Databases   
2    N003  Data Engineering             Data Cleaning   
3    N004  Data Engineering  APIs and Data Collection   
4    N005  Data Engineering      Big Data and PySpark   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
3  An API or Application Programming Interface al...  
4  Big Data refers to extremely large datasets th...  


In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
documents = df["content"].tolist()
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

In [28]:
import chromadb
client = chromadb.Client()
collection = client.get_or_create_collection(
    name="college_notes"
)
ids = [f"note_{i}" for i in range(len(documents))]
metadatas = []
for _, row in df.iterrows():
    metadatas.append({
        "subject": row["subject"],
        "topic": row["topic"]
    })
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)
print("Documents stored:", collection.count())

Documents stored: 15


In [29]:
def retrieve_notes(question, top_k=3):
    question_embedding = embedding_model.encode(
        question
    ).tolist()
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )
    return results

In [30]:
def build_context(results):
    context = ""
    for i, (doc, meta) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0]
        )
    ):
        context += (
            f"[Source {i+1}: "
            f"{meta['subject']} - "
            f"{meta['topic']}]\n"
            f"{doc}\n\n"
        )
    return context

In [ ]:
from groq import Groq
import os
client_groq = Groq(
    api_key=os.getenv("YOUR_GROQ_API_KEY")
)

In [32]:
def generate_answer(question, context):
    system_prompt = """
You are a College Knowledge Assistant.
Rules:
1. Use only the provided context.
2. If answer is not present, say:
   'I don't have enough information
   in my knowledge base.'
3. Mention sources used.
"""
    user_prompt = f"""
Context:
{context}
Question:
{question}
"""
    response = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role":"system",
                "content":system_prompt
            },
            {
                "role":"user",
                "content":user_prompt
            }
        ],
        temperature=0.1
    )
    return response.choices[0].message.content

In [33]:
def ask_college_assistant(
        question,
        top_k=3
):
    print("="*60)
    print("Retrieving relevant notes...")
    results = retrieve_notes(
        question,
        top_k
    )
    for i, meta in enumerate(
        results["metadatas"][0]
    ):
        print(
            f"{i+1}. "
            f"{meta['subject']} - "
            f"{meta['topic']}"
        )
    context = build_context(results)
    answer = generate_answer(
        question,
        context
    )
    print("-"*60)
    print("ANSWER:")
    print(answer)
    print("="*60)
    return answer

In [34]:
ask_college_assistant(
    "What is the population of Tokyo?"
)

Retrieving relevant notes...
1. Generative AI - Large Language Models
2. Data Engineering - SQL Databases
3. Data Engineering - Data Cleaning
------------------------------------------------------------
ANSWER:
I don't have enough information in my knowledge base to provide the current population of Tokyo.


"I don't have enough information in my knowledge base to provide the current population of Tokyo."

--END OF MINIPROJECT DAY-8--